### Structured Output in LangChain
Models can be instructed to return responses in a specific format that matches a predefined schema. This makes the output easier to validate, parse, and use in subsequent processing.
##### Key Points
- Schema: Defines the expected structure, fields, and data types of the output.
- Validation: Ensures the returned data follows the required format.
- Parsing: Converts the model's response into a structured representation, such as a Python object.
- Use cases: API responses, data extraction, information processing, and automation.
- LangChain supports multiple approaches to structured output, depending on the model and integration.



#### Pydantic
Pydantic is a Python library used to define data schemas and validate data using Python type hints.
##### Key Features
- Field validation: Checks whether values match the expected types and constraints.
- Descriptions: Adds explanations to fields to clarify their purpose.
- Nested structures: Supports complex data models containing other models.
- Type safety: Helps ensure data follows the defined schema.

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The rating of the movie out of 10")
    

In [4]:
model_with_structure = model.with_structured_output(Movie)

In [5]:
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000200DE3D1E80>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000200DE3D2900>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The ti

In [6]:
model.invoke("Provide details about the movie Inception")

AIMessage(content='**Inception (2010) – Quick Reference Guide**\n\n| Item | Details |\n|------|---------|\n| **Title** | *Inception* |\n| **Director** | Christopher Nolan |\n| **Screenwriter** | Christopher Nolan (story & screenplay) |\n| **Producer(s)** | Emma Thomas, Christopher Nolan, Charles Roven, Jordan Goldberg |\n| **Cinematographer** | Wally Pfister |\n| **Music** | Hans Zimmer |\n| **Editor** | Lee Smith |\n| **Production Companies** | Legendary Pictures, Syncopy Inc. |\n| **Distributor** | Warner Bros. Pictures |\n| **Release Dates** | • World Premiere – July\u202f8,\u202f2010 (London)  <br>• U.S. Theatrical Release – July\u202f16,\u202f2010 |\n| **Running Time** | 148 minutes (2\u202fh\u202f28\u202fmin) |\n| **Budget** | Approx. **$160\u202fmillion** (including marketing) |\n| **Box‑Office Gross** | • Worldwide: **$839.9\u202fmillion**  <br>• Domestic (U.S./Canada): $292\u202fmillion  <br>• International: $548\u202fmillion |\n| **MPAA Rating** | PG‑13 (Violence, Some Langua

In [8]:
model_with_structure.invoke("Provide details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

#### Raw Message  output alongside parsed output structure

In [15]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="The year the movie was released")
    director:str=Field(...,description="The director of the movie")
    rating:float=Field(...,description="The rating of the movie out of 10")

model_with_structure = model.with_structured_output(Movie,include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks "Provide details about the movie Inception". We need to give details. Could be title, director, rating, year. Use the function Movie. Provide those details. The function expects director, rating, title, year. We have: Inception directed by Christopher Nolan, year 2010, rating maybe 8.8 (IMDb). Provide that. We\'ll call function.', 'tool_calls': [{'id': 'fc_0da6ddac-bd1f-48f2-b7e9-3c4e4b13cdff', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 132, 'prompt_tokens': 165, 'total_tokens': 297, 'completion_time': 0.280351544, 'completion_tokens_details': {'reasoning_tokens': 80}, 'prompt_time': 0.01286189, 'prompt_tokens_details': None, 'queue_time': 0.212645707, 'total_time': 0.293213434}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_86

#### Nested Structure

In [32]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:str
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None,description="Budget in USD")

model_with_structure = model.with_structured_output(MovieDetails)

In [33]:
model_with_structure.invoke("Provide details about the movie Inception")

MovieDetails(title='Inception', year='2010', cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Professor Stephen Miles'), Actor(name='Dileep Rao', role='Yusuf'), Actor(name='Pete Postlethwaite', role='Maurice Fischer')], genres=['Action', 'Adventure', 'Sci-Fi', 'Thriller'], budget=160000000.0)